In [ ]:
# CELL 1: Install Dependencies
!pip install pyscipopt torch torch-geometric numpy scipy matplotlib pandas
!git clone https://gitlab.com/uniluxembourg/snt/pcog/ultrabo/clustering-for-search-strategy
!git clone https://github.com/ds4dm/ecole  # for reference on GNN architecture only, not running ecole

In [ ]:
# CELL 2: Download StrIPLIB Instances
import os
import re
import gzip
import shutil
import tarfile
import zipfile
import tempfile
import urllib.parse
import urllib.request
from pathlib import Path

INSTANCE_ROOT = Path('/content/instances')
STRIPLIB_BASE = 'https://striplib.or.rwth-aachen.de'
CLASSES = ['vrp', 'knapsack', 'bin_packing']
MIN_INSTANCES_PER_CLASS = 50
MAX_VARIABLES = 500

for cls in CLASSES:
    (INSTANCE_ROOT / cls).mkdir(parents=True, exist_ok=True)


def safe_print_error(context, exc):
    try:
        print(f'[ERROR] {context}: {type(exc).__name__}: {exc}')
    except Exception:
        print('[ERROR] could not format exception')


def urlopen_text(url, timeout=20):
    try:
        req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0 Colab MILP experiment'})
        with urllib.request.urlopen(req, timeout=timeout) as resp:
            return resp.read().decode('utf-8', errors='ignore')
    except Exception as exc:
        safe_print_error(f'fetching text {url}', exc)
        return ''


def discover_striplib_links(class_name, max_pages=80):
    """Crawl StrIPLIB lightly and keep downloadable links that mention the target class."""
    try:
        seen_pages = set()
        queue = [STRIPLIB_BASE, f'{STRIPLIB_BASE}/{class_name}', f'{STRIPLIB_BASE}/instances', f'{STRIPLIB_BASE}/instances/{class_name}']
        downloads = []
        wanted_ext = ('.mps', '.mps.gz', '.zip', '.tar.gz', '.tgz')
        class_tokens = {class_name, class_name.replace('_', '-'), class_name.replace('_', '')}

        while queue and len(seen_pages) < max_pages and len(downloads) < MIN_INSTANCES_PER_CLASS * 4:
            page = queue.pop(0)
            if page in seen_pages:
                continue
            seen_pages.add(page)
            html = urlopen_text(page)
            if not html:
                continue
            hrefs = re.findall(r'href=["\']([^"\']+)["\']', html, flags=re.I)
            for href in hrefs:
                absolute = urllib.parse.urljoin(page, href)
                lower = absolute.lower()
                if any(ext in lower for ext in wanted_ext) and any(tok in lower for tok in class_tokens):
                    downloads.append(absolute)
                elif lower.startswith(STRIPLIB_BASE.lower()) and any(tok in lower for tok in class_tokens):
                    if absolute not in seen_pages and absolute not in queue:
                        queue.append(absolute)
        return sorted(set(downloads))
    except Exception as exc:
        safe_print_error(f'discovering links for {class_name}', exc)
        return []


def download_file(url, dest_dir):
    try:
        local = dest_dir / Path(urllib.parse.urlparse(url).path).name
        if local.exists() and local.stat().st_size > 0:
            return local
        req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0 Colab MILP experiment'})
        with urllib.request.urlopen(req, timeout=60) as resp, open(local, 'wb') as out:
            shutil.copyfileobj(resp, out)
        return local
    except Exception as exc:
        safe_print_error(f'downloading {url}', exc)
        return None


def extract_mps_files(archive_path, dest_dir):
    try:
        extracted = []
        name = archive_path.name.lower()
        if name.endswith('.mps'):
            return [archive_path]
        if name.endswith('.mps.gz'):
            out = dest_dir / archive_path.name[:-3]
            with gzip.open(archive_path, 'rb') as src, open(out, 'wb') as dst:
                shutil.copyfileobj(src, dst)
            return [out]
        if name.endswith('.zip'):
            with zipfile.ZipFile(archive_path) as zf:
                for member in zf.namelist():
                    if member.lower().endswith(('.mps', '.mps.gz')):
                        zf.extract(member, dest_dir)
                        extracted.append(dest_dir / member)
        if name.endswith(('.tar.gz', '.tgz')):
            with tarfile.open(archive_path) as tf:
                for member in tf.getmembers():
                    if member.name.lower().endswith(('.mps', '.mps.gz')):
                        tf.extract(member, dest_dir)
                        extracted.append(dest_dir / member.name)
        final_files = []
        for path in extracted:
            path = Path(path)
            if path.name.lower().endswith('.mps.gz'):
                out = path.with_suffix('')
                with gzip.open(path, 'rb') as src, open(out, 'wb') as dst:
                    shutil.copyfileobj(src, dst)
                final_files.append(out)
            else:
                final_files.append(path)
        return final_files
    except Exception as exc:
        safe_print_error(f'extracting {archive_path}', exc)
        return []


def is_small_mps(mps_path, max_variables=MAX_VARIABLES):
    try:
        import pyscipopt
        model = pyscipopt.Model()
        model.hideOutput(True)
        model.readProblem(str(mps_path))
        return len(model.getVars()) < max_variables
    except Exception as exc:
        safe_print_error(f'checking size for {mps_path}', exc)
        return False


def load_striplib_class(class_name):
    try:
        dest = INSTANCE_ROOT / class_name
        links = discover_striplib_links(class_name)
        print(f'{class_name}: discovered {len(links)} candidate download links')
        kept = sorted(dest.glob('*.mps'))
        with tempfile.TemporaryDirectory() as tmp:
            tmpdir = Path(tmp)
            for url in links:
                if len(kept) >= MIN_INSTANCES_PER_CLASS:
                    break
                downloaded = download_file(url, tmpdir)
                if downloaded is None:
                    continue
                for candidate in extract_mps_files(downloaded, tmpdir):
                    if len(kept) >= MIN_INSTANCES_PER_CLASS:
                        break
                    if not candidate.exists() or not candidate.name.lower().endswith('.mps'):
                        continue
                    if not is_small_mps(candidate):
                        continue
                    target = dest / f'{len(kept):03d}_{candidate.name}'
                    shutil.copy2(candidate, target)
                    kept.append(target)
        print(f'{class_name}: loaded {len(sorted(dest.glob("*.mps")))} small MPS instances into {dest}')
    except Exception as exc:
        safe_print_error(f'loading class {class_name}', exc)


for cls in CLASSES:
    load_striplib_class(cls)

for cls in CLASSES:
    print(f'{cls}: {len(list((INSTANCE_ROOT / cls).glob("*.mps")))} instances loaded')

In [ ]:
# CELL 3: Distance Metric Implementation
import math
from collections import Counter, defaultdict
from pathlib import Path
import numpy as np


def safe_print_error(context, exc):
    try:
        print(f'[ERROR] {context}: {type(exc).__name__}: {exc}')
    except Exception:
        print('[ERROR] could not format exception')


def bucket_variable(var):
    try:
        vt = str(var.vtype()).upper()
        lb, ub = float(var.getLbGlobal()), float(var.getUbGlobal())
        if 'BINARY' in vt or (('INTEGER' in vt or 'IMPLINT' in vt) and lb == 0.0 and ub == 1.0):
            return 'B'
        if 'INTEGER' in vt or 'IMPLINT' in vt:
            return 'I'
        return 'C'
    except Exception as exc:
        safe_print_error('bucketing variable', exc)
        return 'C'


def bucket_weight(value):
    try:
        # Section 3.2: singleton classes -1 and 1, all other weights in R.
        if math.isclose(float(value), -1.0, abs_tol=1e-9):
            return '-1'
        if math.isclose(float(value), 1.0, abs_tol=1e-9):
            return '1'
        return 'R'
    except Exception as exc:
        safe_print_error('bucketing weight', exc)
        return 'R'


def bucket_rhs(value):
    try:
        # Section 3.2: singleton classes 0 and 1, all other RHS values in R.
        if math.isclose(float(value), 0.0, abs_tol=1e-9):
            return '0'
        if math.isclose(float(value), 1.0, abs_tol=1e-9):
            return '1'
        return 'R'
    except Exception as exc:
        safe_print_error('bucketing RHS', exc)
        return 'R'


def normalize_pair_distribution(pair_counts):
    try:
        total = sum(pair_counts.values())
        if total <= 0:
            return {}
        return {pair: count / total for pair, count in pair_counts.items()}
    except Exception as exc:
        safe_print_error('normalizing pair distribution', exc)
        return {}


def canonical_constraint_key(constraint_rep):
    try:
        pairs = tuple(sorted((w, v, round(float(p), 8)) for (w, v), p in constraint_rep['pairs'].items()))
        return (pairs, constraint_rep['rhs_class'])
    except Exception as exc:
        safe_print_error('canonicalizing constraint', exc)
        return tuple()


def extract_normalized_representation(mps_path):
    try:
        import pyscipopt
        model = pyscipopt.Model()
        model.hideOutput(True)
        model.readProblem(str(mps_path))
        vars_ = model.getVars()
        var_class = {var.name: bucket_variable(var) for var in vars_}

        objective_counts = Counter()
        for var in vars_:
            coeff = model.getObjCoef(var)
            if abs(float(coeff)) > 1e-12:
                objective_counts[(bucket_weight(coeff), var_class[var.name])] += 1
        objective = {'pairs': normalize_pair_distribution(objective_counts), 'rhs_class': '0'}

        constraints = []
        for cons in model.getConss():
            try:
                vals = model.getValsLinear(cons)
                if not vals:
                    continue
                rhs = model.getRhs(cons)
                lhs = model.getLhs(cons)

                rows = []
                # Section 3.1/3.3: represent constraints as <= rows; ranged/equality rows become two inequalities.
                if rhs < model.infinity():
                    rows.append((vals, rhs))
                if lhs > -model.infinity():
                    rows.append(({name: -coef for name, coef in vals.items()}, -lhs))

                for row_vals, row_rhs in rows:
                    pair_counts = Counter()
                    for name, coef in row_vals.items():
                        var_name = name if isinstance(name, str) else getattr(name, 'name', str(name))
                        if abs(float(coef)) <= 1e-12:
                            continue
                        pair_counts[(bucket_weight(coef), var_class.get(var_name, 'C'))] += 1
                    if pair_counts:
                        # Section 3.3: pc is the proportion of each weight-variable pair inside a constraint.
                        constraints.append({'pairs': normalize_pair_distribution(pair_counts), 'rhs_class': bucket_rhs(row_rhs)})
            except Exception as exc:
                safe_print_error(f'extracting constraint in {mps_path}', exc)
                continue

        counts = Counter(canonical_constraint_key(c) for c in constraints)
        exemplar = {}
        for c in constraints:
            exemplar.setdefault(canonical_constraint_key(c), c)
        m = max(1, len(constraints))
        normalized_constraints = []
        for key, count in counts.items():
            rep = dict(exemplar[key])
            # Section 3.3: pP is the proportion of each unique constraint in the instance.
            rep['proportion'] = count / m
            normalized_constraints.append(rep)

        return {
            'path': str(mps_path),
            'objective': objective,
            'constraints': normalized_constraints,
            'n_constraints': len(constraints),
            'n_unique_constraints': len(normalized_constraints),
            'n_variables': len(vars_),
        }
    except Exception as exc:
        safe_print_error(f'extract_normalized_representation({mps_path})', exc)
        return None


def pair_distance(pair1, pair2, alpha=1, beta=1):
    try:
        # Equation 3: weight mismatch costs alpha; variable-class mismatch costs beta.
        return (alpha if pair1[0] != pair2[0] else 0) + (beta if pair1[1] != pair2[1] else 0)
    except Exception as exc:
        safe_print_error('pair distance', exc)
        return float('inf')


def greedy_constraint_distance(c1, c2, alpha=1, beta=1, gamma=1):
    try:
        if c1 is None or c2 is None:
            return float('inf')
        supply = {pair: float(prop) for pair, prop in c1.get('pairs', {}).items()}
        demand = {pair: float(prop) for pair, prop in c2.get('pairs', {}).items()}
        dist = gamma if c1.get('rhs_class') != c2.get('rhs_class') else 0.0

        # Section 3.5 greedy variant of Equation 4: repeatedly move the largest possible mass between closest pairs.
        while supply and demand:
            best = None
            for p1, s in supply.items():
                if s <= 1e-12:
                    continue
                for p2, d in demand.items():
                    if d <= 1e-12:
                        continue
                    cost = pair_distance(p1, p2, alpha=alpha, beta=beta)
                    if best is None or cost < best[0]:
                        best = (cost, p1, p2)
            if best is None:
                break
            cost, p1, p2 = best
            moved = min(supply[p1], demand[p2])
            dist += cost * moved
            supply[p1] -= moved
            demand[p2] -= moved
            if supply[p1] <= 1e-12:
                del supply[p1]
            if demand[p2] <= 1e-12:
                del demand[p2]
        return float(dist)
    except Exception as exc:
        safe_print_error('greedy_constraint_distance', exc)
        return float('inf')


def greedy_instance_distance(rep1, rep2, zeta=1):
    try:
        if rep1 is None or rep2 is None:
            return float('inf')
        c1s = list(rep1.get('constraints', []))
        c2s = list(rep2.get('constraints', []))
        supply = {i: float(c.get('proportion', 0.0)) for i, c in enumerate(c1s)}
        demand = {j: float(c.get('proportion', 0.0)) for j, c in enumerate(c2s)}
        dist = 0.0
        cache = {}

        # Section 3.5 greedy variant of Equation 5: repeatedly match closest constraints by dc.
        while supply and demand:
            best = None
            for i, s in supply.items():
                if s <= 1e-12:
                    continue
                for j, d in demand.items():
                    if d <= 1e-12:
                        continue
                    key = (i, j)
                    if key not in cache:
                        cache[key] = greedy_constraint_distance(c1s[i], c2s[j])
                    cost = cache[key]
                    if best is None or cost < best[0]:
                        best = (cost, i, j)
            if best is None:
                break
            cost, i, j = best
            moved = min(supply[i], demand[j])
            dist += cost * moved
            supply[i] -= moved
            demand[j] -= moved
            if supply[i] <= 1e-12:
                del supply[i]
            if demand[j] <= 1e-12:
                del demand[j]

        # Equation 5: zeta-weighted objective-function distance.
        dist += zeta * greedy_constraint_distance(rep1.get('objective'), rep2.get('objective'))
        return float(dist)
    except Exception as exc:
        safe_print_error('greedy_instance_distance', exc)
        return float('inf')


def compute_distribution_distance(test_rep, reference_reps):
    try:
        distances = [greedy_instance_distance(test_rep, ref) for ref in reference_reps if ref is not None]
        distances = [d for d in distances if np.isfinite(d)]
        return float(np.mean(distances)) if distances else float('inf')
    except Exception as exc:
        safe_print_error('compute_distribution_distance', exc)
        return float('inf')


# Verification requested in the prompt.
try:
    same_class_paths = sorted((Path('/content/instances/vrp')).glob('*.mps'))[:2]
    cross_class_paths = sorted((Path('/content/instances/knapsack')).glob('*.mps'))[:1]
    if len(same_class_paths) >= 2 and cross_class_paths:
        rep_a = extract_normalized_representation(same_class_paths[0])
        rep_b = extract_normalized_representation(same_class_paths[1])
        rep_c = extract_normalized_representation(cross_class_paths[0])
        same_distance = greedy_instance_distance(rep_a, rep_b)
        cross_distance = greedy_instance_distance(rep_a, rep_c)
        print(f'same class distance: {same_distance:.4f}')
        print(f'cross class distance: {cross_distance:.4f}')
        print('verification:', 'PASS' if same_distance < cross_distance else 'CHECK: same-class distance was not lower')
    else:
        print('verification skipped: not enough downloaded instances')
except Exception as exc:
    safe_print_error('distance verification', exc)

In [ ]:
# CELL 4: GNN Branching Model
import random
import numpy as np
import torch
import torch.nn.functional as F
from torch import nn
try:
    from torch_geometric.data import Data
    from torch_geometric.nn import MessagePassing
except Exception as exc:
    safe_print_error('importing torch_geometric', exc)


def finite_or_zero(x):
    try:
        x = float(x)
        if not np.isfinite(x):
            return 0.0
        return x
    except Exception:
        return 0.0


def encode_variable_type(var):
    try:
        cls = bucket_variable(var)
        return {'B': [1.0, 0.0, 0.0], 'I': [0.0, 1.0, 0.0], 'C': [0.0, 0.0, 1.0]}.get(cls, [0.0, 0.0, 1.0])
    except Exception as exc:
        safe_print_error('encoding variable type', exc)
        return [0.0, 0.0, 1.0]


def _model_to_bipartite_graph(model):
    try:
        vars_ = model.getVars()
        conss = model.getConss()
        var_index = {v.name: i for i, v in enumerate(vars_)}
        cons_index = {c.name: i for i, c in enumerate(conss)}

        var_features = []
        for var in vars_:
            try:
                obj = finite_or_zero(model.getObjCoef(var))
                lb = finite_or_zero(var.getLbGlobal())
                ub = finite_or_zero(var.getUbGlobal())
                if abs(lb) > 1e6:
                    lb = np.sign(lb) * 1e6
                if abs(ub) > 1e6:
                    ub = np.sign(ub) * 1e6
                var_features.append([obj, lb, ub] + encode_variable_type(var))
            except Exception as exc:
                safe_print_error(f'variable features for {var}', exc)
                var_features.append([0.0, 0.0, 0.0, 0.0, 0.0, 1.0])

        cons_features = []
        edges = []
        edge_attr = []
        for ci, cons in enumerate(conss):
            try:
                rhs = finite_or_zero(model.getRhs(cons))
                lhs = finite_or_zero(model.getLhs(cons))
                if abs(rhs) > 1e6:
                    rhs = np.sign(rhs) * 1e6
                if abs(lhs) > 1e6:
                    lhs = np.sign(lhs) * 1e6
                sense = [float(model.getLhs(cons) > -model.infinity()), float(model.getRhs(cons) < model.infinity())]
                cons_features.append([lhs, rhs] + sense)
                vals = model.getValsLinear(cons)
                for name, coef in vals.items():
                    var_name = name if isinstance(name, str) else getattr(name, 'name', str(name))
                    if var_name in var_index and abs(float(coef)) > 1e-12:
                        edges.append([ci, var_index[var_name]])
                        edge_attr.append([finite_or_zero(coef)])
            except Exception as exc:
                safe_print_error(f'constraint features for {cons}', exc)
                cons_features.append([0.0, 0.0, 0.0, 1.0])

        graph = Data(
            x_constraints=torch.tensor(cons_features, dtype=torch.float32),
            x_variables=torch.tensor(var_features, dtype=torch.float32),
            edge_index=torch.tensor(edges, dtype=torch.long).t().contiguous() if edges else torch.empty((2, 0), dtype=torch.long),
            edge_attr=torch.tensor(edge_attr, dtype=torch.float32) if edge_attr else torch.empty((0, 1), dtype=torch.float32),
            var_names=[v.name for v in vars_],
        )
        return graph
    except Exception as exc:
        safe_print_error('_model_to_bipartite_graph', exc)
        return None


def milp_to_bipartite_graph(mps_path):
    try:
        import pyscipopt
        model = pyscipopt.Model()
        model.hideOutput(True)
        model.readProblem(str(mps_path))
        return _model_to_bipartite_graph(model)
    except Exception as exc:
        safe_print_error(f'milp_to_bipartite_graph({mps_path})', exc)
        return None


class BipartiteGNN(nn.Module):
    """Lightweight bipartite MILP GNN following the variable/constraint graph idea of Gasse et al. 2019."""
    def __init__(self, var_dim=6, cons_dim=4, edge_dim=1, hidden_dim=64):
        super().__init__()
        try:
            self.var_encoder = nn.Sequential(nn.Linear(var_dim, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, hidden_dim))
            self.cons_encoder = nn.Sequential(nn.Linear(cons_dim, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, hidden_dim))
            self.edge_encoder = nn.Sequential(nn.Linear(edge_dim, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, hidden_dim))
            self.var_update = nn.Sequential(nn.Linear(hidden_dim * 3, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, hidden_dim))
            self.cons_update = nn.Sequential(nn.Linear(hidden_dim * 3, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, hidden_dim))
            self.scorer = nn.Sequential(nn.Linear(hidden_dim, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, 1))
        except Exception as exc:
            safe_print_error('initializing BipartiteGNN', exc)

    def forward(self, data):
        try:
            xv = self.var_encoder(data.x_variables)
            xc = self.cons_encoder(data.x_constraints)
            if data.edge_index.numel() == 0:
                return self.scorer(xv).squeeze(-1)
            ei = data.edge_index
            edge = self.edge_encoder(data.edge_attr)

            # Two rounds of variable-constraint message passing over nonzero matrix entries.
            for _ in range(2):
                c_idx, v_idx = ei[0], ei[1]
                msg_to_var = xc[c_idx] + edge
                agg_var = torch.zeros_like(xv).index_add(0, v_idx, msg_to_var)
                deg_var = torch.zeros((xv.shape[0], 1), device=xv.device).index_add(0, v_idx, torch.ones((v_idx.shape[0], 1), device=xv.device)).clamp_min(1.0)
                agg_var = agg_var / deg_var
                xv = self.var_update(torch.cat([xv, agg_var, xv * agg_var], dim=-1))

                msg_to_cons = xv[v_idx] + edge
                agg_cons = torch.zeros_like(xc).index_add(0, c_idx, msg_to_cons)
                deg_cons = torch.zeros((xc.shape[0], 1), device=xc.device).index_add(0, c_idx, torch.ones((c_idx.shape[0], 1), device=xc.device)).clamp_min(1.0)
                agg_cons = agg_cons / deg_cons
                xc = self.cons_update(torch.cat([xc, agg_cons, xc * agg_cons], dim=-1))

            return self.scorer(xv).squeeze(-1)
        except Exception as exc:
            safe_print_error('BipartiteGNN.forward', exc)
            n = getattr(data, 'x_variables', torch.empty((0, 6))).shape[0]
            return torch.zeros(n)


class MLBranchingRule(__import__('pyscipopt').Branchrule):
    def __init__(self, gnn_model):
        super().__init__()
        try:
            self.gnn_model = gnn_model
            self.gnn_model.eval()
        except Exception as exc:
            safe_print_error('initializing MLBranchingRule', exc)

    def branchexeclp(self, allowaddcons):
        try:
            import pyscipopt
            cands = self.model.getLPBranchCands()[0]
            if not cands:
                return {'result': pyscipopt.SCIP_RESULT.DIDNOTRUN}
            graph = _model_to_bipartite_graph(self.model)
            if graph is None:
                return {'result': pyscipopt.SCIP_RESULT.DIDNOTRUN}
            with torch.no_grad():
                scores = self.gnn_model(graph)
            name_to_idx = {name: i for i, name in enumerate(graph.var_names)}
            best_var = None
            best_score = -float('inf')
            for var in cands:
                idx = name_to_idx.get(var.name)
                if idx is None:
                    continue
                score = float(scores[idx].item())
                if score > best_score:
                    best_var, best_score = var, score
            if best_var is None:
                return {'result': pyscipopt.SCIP_RESULT.DIDNOTRUN}
            self.model.branchVar(best_var)
            return {'result': pyscipopt.SCIP_RESULT.BRANCHED}
        except Exception as exc:
            safe_print_error('MLBranchingRule.branchexeclp', exc)
            try:
                import pyscipopt
                return {'result': pyscipopt.SCIP_RESULT.DIDNOTRUN}
            except Exception:
                return {'result': None}


def collect_strong_branching_labels(mps_path):
    try:
        import pyscipopt
        model = pyscipopt.Model()
        model.hideOutput(True)
        model.setParam('limits/time', 60)
        model.readProblem(str(mps_path))
        graph = _model_to_bipartite_graph(model)
        if graph is None:
            return None, None, False

        labels = torch.zeros(graph.x_variables.shape[0], dtype=torch.float32)
        used_strong_branching = False
        name_to_idx = {name: i for i, name in enumerate(graph.var_names)}
        try:
            model.setParam('presolving/maxrounds', 0)
            model.presolve()
            model.constructLP()
            cands = model.getLPBranchCands()[0]
            for var in cands:
                try:
                    result = model.getVarStrongbranch(var, 100, True)
                    numeric = [float(x) for x in result if isinstance(x, (int, float)) and np.isfinite(float(x))]
                    score = max(numeric) if numeric else 0.0
                    labels[name_to_idx[var.name]] = score
                    used_strong_branching = True
                except Exception:
                    continue
        except Exception as exc:
            safe_print_error(f'strong branching failed for {mps_path}; using pseudo-labels', exc)

        if not used_strong_branching:
            # Explicit fallback requested: random pseudo-labels if strong branching collection is too slow/unavailable.
            print(f'NOTE: using random branching pseudo-labels for {Path(mps_path).name}')
            labels = torch.rand(graph.x_variables.shape[0], dtype=torch.float32)
        return graph, labels, used_strong_branching
    except Exception as exc:
        safe_print_error(f'collect_strong_branching_labels({mps_path})', exc)
        return None, None, False


def train_gnn(instance_paths, n_epochs=20):
    try:
        model = BipartiteGNN()
        optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
        dataset = []
        strong_count = 0
        for path in instance_paths:
            graph, labels, used_strong = collect_strong_branching_labels(path)
            if graph is not None and labels is not None and labels.numel() == graph.x_variables.shape[0]:
                dataset.append((graph, labels))
                strong_count += int(used_strong)
        print(f'training examples: {len(dataset)}; strong-branching labels: {strong_count}; pseudo-label examples: {len(dataset) - strong_count}')
        if not dataset:
            print('No training data collected; returning randomly initialized model')
            return model
        for epoch in range(n_epochs):
            losses = []
            random.shuffle(dataset)
            for graph, labels in dataset:
                try:
                    optimizer.zero_grad()
                    pred = model(graph)
                    target = (labels - labels.mean()) / (labels.std().clamp_min(1e-6))
                    loss = F.mse_loss(pred, target)
                    loss.backward()
                    optimizer.step()
                    losses.append(float(loss.item()))
                except Exception as exc:
                    safe_print_error('training step', exc)
                    continue
            if (epoch + 1) % 5 == 0 or epoch == 0:
                print(f'epoch {epoch + 1}/{n_epochs}: loss={np.mean(losses) if losses else float("nan"):.4f}')
        return model
    except Exception as exc:
        safe_print_error('train_gnn', exc)
        return BipartiteGNN()

In [ ]:
# CELL 5: Branching Evaluation
def evaluate_branching(model, mps_path, time_limit=60):
    try:
        import pyscipopt

        def run_default():
            try:
                scip = pyscipopt.Model()
                scip.hideOutput(True)
                scip.setParam('limits/time', float(time_limit))
                scip.readProblem(str(mps_path))
                scip.optimize()
                return int(scip.getNNodes())
            except Exception as exc:
                safe_print_error(f'default SCIP solve {mps_path}', exc)
                return None

        def run_ml():
            try:
                ml = pyscipopt.Model()
                ml.hideOutput(True)
                ml.setParam('limits/time', float(time_limit))
                ml.readProblem(str(mps_path))
                rule = MLBranchingRule(model)
                ml.includeBranchrule(rule, 'ml_branching', 'GNN branching rule', priority=1000000, maxdepth=-1, maxbounddist=1.0)
                ml.optimize()
                return int(ml.getNNodes())
            except Exception as exc:
                safe_print_error(f'ML SCIP solve {mps_path}', exc)
                return None

        scip_nodes = run_default()
        ml_nodes = run_ml()
        if scip_nodes is None or ml_nodes is None:
            return None
        return {
            'ml_nodes': int(ml_nodes),
            'scip_nodes': int(scip_nodes),
            'degradation': float(ml_nodes / max(1, scip_nodes)),
        }
    except Exception as exc:
        safe_print_error(f'evaluate_branching({mps_path})', exc)
        return None

In [ ]:
# CELL 6: Core Experiment Pipeline
import pandas as pd
from pathlib import Path

training_class = 'vrp'
test_classes = ['vrp', 'knapsack', 'bin_packing']
RESULTS_DIR = Path('/content/results')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

try:
    training_paths = sorted((Path('/content/instances') / training_class).glob('*.mps'))
    reference_paths = training_paths[:40]
    train_paths = training_paths[:40]
    print(f'reference instances: {len(reference_paths)} from {training_class}')

    reference_reps = []
    for i, path in enumerate(reference_paths):
        rep = extract_normalized_representation(path)
        if rep is not None:
            reference_reps.append(rep)
        if (i + 1) % 10 == 0:
            print(f'precomputed {i + 1}/{len(reference_paths)} reference representations')

    gnn_model = train_gnn(train_paths, n_epochs=20)

    rows = []
    processed = 0
    for cls in test_classes:
        paths = sorted((Path('/content/instances') / cls).glob('*.mps'))[:MIN_INSTANCES_PER_CLASS]
        print(f'evaluating {len(paths)} instances for class {cls}')
        for path in paths:
            try:
                rep = extract_normalized_representation(path)
                if rep is None:
                    continue
                distance = compute_distribution_distance(rep, reference_reps)
                eval_result = evaluate_branching(gnn_model, path, time_limit=60)
                if eval_result is None:
                    continue
                rows.append({
                    'instance_name': path.name,
                    'class': cls,
                    'distance': distance,
                    'ml_nodes': eval_result['ml_nodes'],
                    'scip_nodes': eval_result['scip_nodes'],
                    'degradation': eval_result['degradation'],
                })
                processed += 1
                if processed % 10 == 0:
                    print(f'processed {processed} instances')
            except Exception as exc:
                safe_print_error(f'pipeline instance {path}', exc)
                continue

    results_df = pd.DataFrame(rows, columns=['instance_name', 'class', 'distance', 'ml_nodes', 'scip_nodes', 'degradation'])
    results_df.to_csv(RESULTS_DIR / 'raw_results.csv', index=False)
    print(f'saved {len(results_df)} rows to {RESULTS_DIR / "raw_results.csv"}')
    display(results_df.head())
except Exception as exc:
    safe_print_error('core experiment pipeline', exc)

In [ ]:
# CELL 7: Results and Plots
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr
from pathlib import Path

RESULTS_DIR = Path('/content/results')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

try:
    results_path = RESULTS_DIR / 'raw_results.csv'
    results_df = pd.read_csv(results_path)
    results_df = results_df.replace([np.inf, -np.inf], np.nan).dropna(subset=['distance', 'degradation', 'class'])
    colors = {'vrp': 'blue', 'knapsack': 'orange', 'bin_packing': 'green'}

    plt.figure(figsize=(9, 6))
    for cls, group in results_df.groupby('class'):
        plt.scatter(group['distance'], group['degradation'], label=cls, color=colors.get(cls, 'gray'), alpha=0.75)
    plt.axhline(1.0, color='black', linestyle='--', linewidth=1, label='ML = SCIP')
    if len(results_df) >= 2:
        x = results_df['distance'].to_numpy(dtype=float)
        y = results_df['degradation'].to_numpy(dtype=float)
        slope, intercept = np.polyfit(x, y, 1)
        xs = np.linspace(x.min(), x.max(), 100)
        plt.plot(xs, slope * xs + intercept, color='red', linewidth=2, label='linear fit')
    plt.xlabel('Distribution distance to VRP training set')
    plt.ylabel('Degradation ratio (ML nodes / SCIP nodes)')
    plt.legend()
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / 'scatter.png', dpi=200)
    plt.show()

    if len(results_df) >= 2:
        corr, pval = pearsonr(results_df['distance'], results_df['degradation'])
        print(f'pearson correlation distance vs degradation: r={corr:.4f}, p={pval:.4g}')
    else:
        print('pearson correlation skipped: fewer than 2 result rows')

    best_threshold = None
    best_accuracy = -1.0
    try:
        labels = (results_df['degradation'] > 1.0).astype(int).to_numpy()
        candidates = sorted(results_df['distance'].unique())
        for threshold in candidates:
            pred = (results_df['distance'].to_numpy() > threshold).astype(int)
            accuracy = float((pred == labels).mean())
            if accuracy > best_accuracy:
                best_accuracy = accuracy
                best_threshold = float(threshold)
        print(f'best distance threshold separating degradation > 1: {best_threshold:.4f} (accuracy={best_accuracy:.3f})')
        for cls, group in results_df.groupby('class'):
            above = int((group['distance'] > best_threshold).sum())
            below = int((group['distance'] <= best_threshold).sum())
            print(f'{cls}: above threshold={above}, below/equal threshold={below}')
    except Exception as exc:
        safe_print_error('threshold analysis', exc)

    plt.figure(figsize=(8, 6))
    order = ['vrp', 'knapsack', 'bin_packing']
    data = [results_df.loc[results_df['class'] == cls, 'degradation'].to_numpy() for cls in order]
    plt.boxplot(data, labels=order, patch_artist=True)
    plt.axhline(1.0, color='black', linestyle='--', linewidth=1)
    plt.ylabel('Degradation ratio (ML nodes / SCIP nodes)')
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / 'boxplot.png', dpi=200)
    plt.show()
    print(f'saved plots to {RESULTS_DIR / "scatter.png"} and {RESULTS_DIR / "boxplot.png"}')
except Exception as exc:
    safe_print_error('results and plots', exc)